In [1]:
import numpy as np
import time

def get_prime_sequence(N):
    """生成真实的素数符号序列 (0: L/存活, 1: R/被筛除)"""
    is_prime = np.ones(N, dtype=bool)
    is_prime[:2] = False
    for i in range(2, int(N**0.5) + 1):
        if is_prime[i]:
            is_prime[i*i:N:i] = False
            
    seq = np.ones(N, dtype=np.int8) # 默认全是 R (1)
    seq[is_prime] = 0               # 质数标记为 L (0)
    seq[1] = 0                      # 根据论文，数字1初始存活
    return seq

def calculate_defect_rate(seq):
    """
    【核心】MSS 奇偶拓扑审查系统 
    扫描序列的所有移位，计算发生拓扑崩溃（原序列 < 移位序列）的次数
    """
    N = len(seq)
    defect_count = 0
    
    # 预计算累加和，O(1) 极速获取任意前缀中 R(1) 的数量
    cumsum_R = np.cumsum(seq)
    
    for shift in range(1, N):
        orig_sub = seq[:N-shift]
        shift_sub = seq[shift:]
        
        # 寻找第一个分歧点
        diffs = (orig_sub != shift_sub)
        if not np.any(diffs):
            continue
            
        first_diff = np.argmax(diffs)
        
        # 获取公共前缀中 'R' 的个数
        r_count = cumsum_R[first_diff - 1] if first_diff > 0 else 0
        
        orig_val = orig_sub[first_diff]
        shift_val = shift_sub[first_diff]
        
        # 自然顺序 L(0) < R(1)
        base_cmp = -1 if orig_val < shift_val else 1
        
        # 奇数个 R 导致拓扑奇偶法则反转！
        real_cmp = -base_cmp if (r_count % 2 == 1) else base_cmp
            
        # 如果最终比较结果是 -1 (原序列 < 移位序列)，说明发生打破最大化条件的拓扑逃逸！
        if real_cmp == -1:
            defect_count += 1
            
    return defect_count, defect_count / (N - 1)

def run_experiment():
    print("🚀 启动实验：真实素数轨道的渐近合法性 (Asymptotic Admissibility)\n")
    N_list = [1000, 2000, 5000, 10000, 20000, 50000] # 你可以往上加到 100000
    
    print(f"{'视界 N':<10} | {'拓扑缺陷数':<12} | {'缺陷密度 ρ(N)':<15} | {'耗时(s)':<10}")
    print("-" * 60)
    
    for N in N_list:
        t0 = time.time()
        seq = get_prime_sequence(N)
        defects, density = calculate_defect_rate(seq)
        t1 = time.time()
        
        print(f"{N:<10} | {defects:<12} | {density:.6f}        | {t1-t0:.2f}")
        
    print("\n✅ 物理预言：你应该会看到密度 ρ(N) 呈现出极其完美的严格衰减趋势！")

if __name__ == "__main__":
    run_experiment()

🚀 启动实验：真实素数轨道的渐近合法性 (Asymptotic Admissibility)

视界 N       | 拓扑缺陷数        | 缺陷密度 ρ(N)       | 耗时(s)     
------------------------------------------------------------
1000       | 0            | 0.000000        | 0.01
2000       | 0            | 0.000000        | 0.08
5000       | 0            | 0.000000        | 0.02
10000      | 0            | 0.000000        | 0.04
20000      | 0            | 0.000000        | 0.08
50000      | 0            | 0.000000        | 0.23

✅ 物理预言：你应该会看到密度 ρ(N) 呈现出极其完美的严格衰减趋势！
